# Data Mapping & Schema Validation
**Cloud Resource Optimisation Thesis**

## Purpose
Validate the actual column structure of all Alibaba trace CSV files against the official schema documentation and correct any mapping errors from Stage 1.

## Files to map
1. `batch_task.csv`
2. `batch_instance.csv` 
3. `container_usage.csv`
4. `container_event.csv` (if needed)
5. `server_usage.csv` (if needed)

---

## Official Schema Reference
Source: `/mnt/project/` schema file from Alibaba Cluster Data repository

We will load each file **without headers** and map columns to the official schema.

In [7]:
import pandas as pd
import os
import numpy as np
from pathlib import Path

In [8]:
# Check your actual project structure
project_root = Path.cwd().parent  # Assumes notebook is in notebooks/
print(f"Current working directory: {Path.cwd()}")
print(f"Project root (guessed): {project_root}")

Current working directory: /Users/prarthanagovindaraj/Desktop/Cloud_Resource_Optimisation_thesis/notebooks
Project root (guessed): /Users/prarthanagovindaraj/Desktop/Cloud_Resource_Optimisation_thesis


In [9]:
# Paths - files are in /mnt/project/
PROJECT_FILES = Path('/Users/prarthanagovindaraj/Desktop/Cloud_Resource_Optimisation_thesis/data/raw/clusterdata2018')

In [10]:
# Load batch_task.csv WITHOUT headers (header=None)
bt_raw = pd.read_csv(PROJECT_FILES / 'batch_task.csv', header=None, nrows=100)

In [11]:
# Load batch_task.csv WITHOUT headers (header=None)
bt_raw = pd.read_csv(PROJECT_FILES / 'batch_task.csv', header=None, nrows=100)

print("=== batch_task.csv — Raw Structure ===")
print(f"Shape: {bt_raw.shape}")
print(f"Number of columns detected: {len(bt_raw.columns)}")
print(f"\nFirst 10 rows:")
print(bt_raw.head(10))
print(f"\nData types per column:")
print(bt_raw.dtypes)
print(f"\nNull counts:")
print(bt_raw.isnull().sum())

=== batch_task.csv — Raw Structure ===
Shape: (100, 8)
Number of columns detected: 8

First 10 rows:
       0      1   2   3      4           5     6         7
0   6459   6524   3   4  15740  Terminated  50.0  0.007957
1   6457   6533   3   5      1  Terminated  50.0  0.004395
2   6036   6046   4   7    393     Waiting   NaN       NaN
3   6036   6046   4   6    452     Waiting   NaN       NaN
4  10719  11332  15  67   1705  Terminated  50.0  0.005736
5  10718  11164  15  66    631  Terminated  50.0  0.016007
6  10718  10916  15  65    300  Terminated  50.0  0.018658
7  10718  12897  15  64   2003  Terminated  50.0  0.016007
8  11792  11999  18  88    257  Terminated  50.0  0.013356
9  11792  14331  18  82   1559  Terminated  50.0  0.016007

Data types per column:
0      int64
1      int64
2      int64
3      int64
4      int64
5        str
6    float64
7    float64
dtype: object

Null counts:
0    0
1    0
2    0
3    0
4    0
5    0
6    2
7    2
dtype: int64


---
## batch_task.csv — Official Schema Mapping

**Official schema from Alibaba documentation:**

| Col | Field Name | Type | Description |
|-----|------------|------|-------------|
| 0 | task_name | string | Task name. Unique within a job |
| 1 | instance_num | bigint | Number of instances |
| 2 | job_name | string | Job name |
| 3 | task_type | string | Task type |
| 4 | status | string | Task status |
| 5 | start_time | bigint | Start time of the task |
| 6 | end_time | bigint | End time of the task |
| 7 | plan_cpu | double | Number of CPU needed, **100 = 1 core** |
| 8 | plan_mem | double | Normalized memory size [0, 100] |

**❌ CRITICAL ERROR DETECTED:**
The file has **8 columns** but the schema shows **9 fields**. 

In [12]:
# Check which columns are actually present by examining the data
print("Checking column mapping against sample data:")
print(f"\nColumn 0 (task_name): {bt_raw[0].dtype} — Sample: {bt_raw[0].iloc[0]}")
print(f"Column 1 (instance_num): {bt_raw[1].dtype} — Sample: {bt_raw[1].iloc[0]}")
print(f"Column 2 (job_name): {bt_raw[2].dtype} — Sample: {bt_raw[2].iloc[0]}")
print(f"Column 3 (task_type): {bt_raw[3].dtype} — Sample: {bt_raw[3].iloc[0]}")
print(f"Column 4 (start_time?): {bt_raw[4].dtype} — Sample: {bt_raw[4].iloc[0]}")
print(f"Column 5 (status): {bt_raw[5].dtype} — Sample: {bt_raw[5].iloc[0]}")
print(f"Column 6 (plan_cpu?): {bt_raw[6].dtype} — Sample: {bt_raw[6].iloc[0]}")
print(f"Column 7 (plan_mem?): {bt_raw[7].dtype} — Sample: {bt_raw[7].iloc[0]}")

print(f"\n\n🔍 Hypothesis: Either end_time is missing, OR the column order is different.")
print(f"Looking at the data pattern:")
print(f"  - Column 4 has value {bt_raw[4].iloc[0]} (likely a timestamp = start_time)")
print(f"  - Column 5 has '{bt_raw[5].iloc[0]}' (definitely status)")
print(f"  - Column 6 has {bt_raw[6].iloc[0]} (50.0 = 0.5 cores if plan_cpu uses 100=1core)")
print(f"  - Column 7 has {bt_raw[7].iloc[0]} (very small = likely plan_mem normalized)")

Checking column mapping against sample data:

Column 0 (task_name): int64 — Sample: 6459
Column 1 (instance_num): int64 — Sample: 6524
Column 2 (job_name): int64 — Sample: 3
Column 3 (task_type): int64 — Sample: 4
Column 4 (start_time?): int64 — Sample: 15740
Column 5 (status): str — Sample: Terminated
Column 6 (plan_cpu?): float64 — Sample: 50.0
Column 7 (plan_mem?): float64 — Sample: 0.0079569280149095


🔍 Hypothesis: Either end_time is missing, OR the column order is different.
Looking at the data pattern:
  - Column 4 has value 15740 (likely a timestamp = start_time)
  - Column 5 has 'Terminated' (definitely status)
  - Column 6 has 50.0 (50.0 = 0.5 cores if plan_cpu uses 100=1core)
  - Column 7 has 0.0079569280149095 (very small = likely plan_mem normalized)


---
## ✅ CORRECTED batch_task.csv Schema

**Actual file structure (8 columns, end_time is missing):**

| Col | Field Name | Type | Description | Stage 1 Error |
|-----|------------|------|-------------|---------------|
| 0 | task_name | int64 | Task name. Unique within a job | ✅ Correct |
| 1 | instance_num | int64 | Number of instances spawned | ❌ Was mapped to job_name |
| 2 | job_name | int64 | Job name (**JOIN KEY**) | ✅ Correct |
| 3 | task_type | int64 | Task type (enumerated) | ❌ Was called task_id |
| 4 | start_time | int64 | Start timestamp in seconds | ✅ Correct |
| 5 | status | string | Task status (Terminated/Waiting) | ✅ Correct |
| 6 | plan_cpu | float64 | **Requested CPU. 100 = 1 core** | ✅ Correct |
| 7 | plan_mem | float64 | Requested memory, normalized [0, 100] | ✅ Correct |

**🔴 Critical Issue Identified:**
- Stage 1 incorrectly assumed column 1 was `job_name`
- The actual `job_name` is in column 2
- Column 1 is `instance_num` (how many instances this task spawned)

In [13]:
# Apply correct column names
BATCH_TASK_SCHEMA = {
    0: 'task_name',
    1: 'instance_num',
    2: 'job_name',
    3: 'task_type', 
    4: 'start_time',
    5: 'status',
    6: 'plan_cpu',
    7: 'plan_mem'
}

bt_mapped = bt_raw.rename(columns=BATCH_TASK_SCHEMA)

print("=== batch_task.csv with CORRECT schema ===")
print(bt_mapped.head(10))
print(f"\n✅ Mapped {len(BATCH_TASK_SCHEMA)} columns")
print(f"📊 Total rows in sample: {len(bt_mapped)}")
print(f"\nColumn types:")
print(bt_mapped.dtypes)

=== batch_task.csv with CORRECT schema ===
   task_name  instance_num  job_name  task_type  start_time      status  \
0       6459          6524         3          4       15740  Terminated   
1       6457          6533         3          5           1  Terminated   
2       6036          6046         4          7         393     Waiting   
3       6036          6046         4          6         452     Waiting   
4      10719         11332        15         67        1705  Terminated   
5      10718         11164        15         66         631  Terminated   
6      10718         10916        15         65         300  Terminated   
7      10718         12897        15         64        2003  Terminated   
8      11792         11999        18         88         257  Terminated   
9      11792         14331        18         82        1559  Terminated   

   plan_cpu  plan_mem  
0      50.0  0.007957  
1      50.0  0.004395  
2       NaN       NaN  
3       NaN       NaN  
4      50.0

---
## batch_instance.csv — Schema Mapping

**Official schema from Alibaba documentation (14 fields expected):**

| Field | Type | Description |
|-------|------|-------------|
| instance_name | string | Instance name |
| task_name | string | Task to which instance belongs |
| job_name | string | Job to which instance belongs |
| task_type | string | Task type |
| status | string | Instance status |
| start_time | bigint | Start time |
| end_time | bigint | End time |
| machine_id | string | Host machine ID |
| seq_no | bigint | Sequence number |
| total_seq_no | bigint | Total sequence number |
| cpu_avg | double | **Average CPU used. 100 = 1 core** |
| cpu_max | double | **Max CPU used. 100 = 1 core** |
| mem_avg | double | Average memory used (normalized) |
| mem_max | double | Max memory used [0, 100] |

Let's validate against actual data...

In [14]:
# Load batch_instance.csv WITHOUT headers
bi_raw = pd.read_csv(PROJECT_FILES / 'batch_instance.csv', header=None, nrows=100)

print("=== batch_instance.csv — Raw Structure ===")
print(f"Shape: {bi_raw.shape}")
print(f"Number of columns detected: {len(bi_raw.columns)}")
print(f"\nFirst 10 rows:")
print(bi_raw.head(10))
print(f"\nData types:")
print(bi_raw.dtypes)
print(f"\nNull counts:")
print(bi_raw.isnull().sum())

=== batch_instance.csv — Raw Structure ===
Shape: (100, 12)
Number of columns detected: 12

First 10 rows:
      0      1    2    3       4           5   6   7     8     9   10  11
0  41562  41618  120  686   299.0  Terminated   1   1  1.50  0.29 NaN NaN
1  41561  41619  120  686  1279.0  Terminated   1   1  0.89  0.28 NaN NaN
2  41562  41617  120  686   828.0  Terminated   1   1  0.94  0.29 NaN NaN
3  41561  41617  120  686    95.0  Terminated   1   1  1.00  0.31 NaN NaN
4  41557  41610  120  686   545.0  Terminated   1   1  1.37  0.29 NaN NaN
5  41557  41614  120  686   258.0  Terminated   1   1  1.18  0.27 NaN NaN
6  41558  41614  120  686   495.0  Terminated   1   1  1.00  0.27 NaN NaN
7  41560  41619  120  686   831.0  Terminated   1   1  0.84  0.37 NaN NaN
8  41561  41616  120  686  1169.0  Terminated   1   1  0.89  0.32 NaN NaN
9  41561  41616  120  686   678.0  Terminated   1   1  0.96  0.30 NaN NaN

Data types:
0       int64
1       int64
2       int64
3       int64
4     floa

In [15]:
# Check column mapping against sample data
print("Checking batch_instance.csv column mapping:")
print(f"\nColumn 0: {bi_raw[0].dtype} — Sample: {bi_raw[0].iloc[0]} (instance_name)")
print(f"Column 1: {bi_raw[1].dtype} — Sample: {bi_raw[1].iloc[0]} (task_name)")
print(f"Column 2: {bi_raw[2].dtype} — Sample: {bi_raw[2].iloc[0]} (job_name)")
print(f"Column 3: {bi_raw[3].dtype} — Sample: {bi_raw[3].iloc[0]} (task_type)")
print(f"Column 4: {bi_raw[4].dtype} — Sample: {bi_raw[4].iloc[0]} (start_time or end_time?)")
print(f"Column 5: {bi_raw[5].dtype} — Sample: '{bi_raw[5].iloc[0]}' (status)")
print(f"Column 6: {bi_raw[6].dtype} — Sample: {bi_raw[6].iloc[0]} (machine_id? or seq_no?)")
print(f"Column 7: {bi_raw[7].dtype} — Sample: {bi_raw[7].iloc[0]} (seq_no?)")
print(f"Column 8: {bi_raw[8].dtype} — Sample: {bi_raw[8].iloc[0]} (cpu_avg?)")
print(f"Column 9: {bi_raw[9].dtype} — Sample: {bi_raw[9].iloc[0]} (cpu_max?)")
print(f"Column 10: {bi_raw[10].dtype} — Sample: {bi_raw[10].iloc[0]} (mem_avg - ALL NULL)")
print(f"Column 11: {bi_raw[11].dtype} — Sample: {bi_raw[11].iloc[0]} (mem_max - ALL NULL)")

print("\n\n🔍 Analysis:")
print(f"- Column 4 = {bi_raw[4].iloc[0]} → Likely start_time (timestamp)")
print(f"- Column 5 = '{bi_raw[5].iloc[0]}' → Definitely status")
print(f"- Column 6 = {bi_raw[6].iloc[0]} → Too small for machine_id, likely seq_no")
print(f"- Column 7 = {bi_raw[7].iloc[0]} → Likely total_seq_no")
print(f"- Column 8 = {bi_raw[8].iloc[0]} → CPU value (1.50 cores = cpu_avg)")
print(f"- Column 9 = {bi_raw[9].iloc[0]} → CPU value (0.29 cores = cpu_max)")
print(f"- Columns 10-11 = ALL NULL (mem_avg, mem_max)")

print("\n❌ MISSING FIELDS:")
print("  - end_time (expected between start_time and status)")
print("  - machine_id (expected after status)")

Checking batch_instance.csv column mapping:

Column 0: int64 — Sample: 41562 (instance_name)
Column 1: int64 — Sample: 41618 (task_name)
Column 2: int64 — Sample: 120 (job_name)
Column 3: int64 — Sample: 686 (task_type)
Column 4: float64 — Sample: 299.0 (start_time or end_time?)
Column 5: str — Sample: 'Terminated' (status)
Column 6: int64 — Sample: 1 (machine_id? or seq_no?)
Column 7: int64 — Sample: 1 (seq_no?)
Column 8: float64 — Sample: 1.5 (cpu_avg?)
Column 9: float64 — Sample: 0.29 (cpu_max?)
Column 10: float64 — Sample: nan (mem_avg - ALL NULL)
Column 11: float64 — Sample: nan (mem_max - ALL NULL)


🔍 Analysis:
- Column 4 = 299.0 → Likely start_time (timestamp)
- Column 5 = 'Terminated' → Definitely status
- Column 6 = 1 → Too small for machine_id, likely seq_no
- Column 7 = 1 → Likely total_seq_no
- Column 8 = 1.5 → CPU value (1.50 cores = cpu_avg)
- Column 9 = 0.29 → CPU value (0.29 cores = cpu_max)
- Columns 10-11 = ALL NULL (mem_avg, mem_max)

❌ MISSING FIELDS:
  - end_time 

---
## ✅ CORRECTED batch_instance.csv Schema

**Actual file structure (12 columns, missing end_time and machine_id):**

| Col | Field Name | Type | Description | Stage 1 Error |
|-----|------------|------|-------------|---------------|
| 0 | instance_name | int64 | Instance identifier | ✅ Correct |
| 1 | task_name | int64 | Task this instance belongs to | ✅ Correct |
| 2 | job_name | int64 | Job this instance belongs to (**JOIN KEY**) | ✅ Correct |
| 3 | task_type | int64 | Task type (enumerated) | ❌ Was called task_id |
| 4 | start_time | float64 | Start timestamp in seconds | ✅ Correct |
| 5 | status | string | Instance status (Terminated/Waiting) | ✅ Correct |
| 6 | seq_no | int64 | Sequence number of this instance | ❌ Was called machine_id |
| 7 | total_seq_no | int64 | Total sequence number | ❌ NEW - was missing |
| 8 | cpu_avg | float64 | **Actual avg CPU used. 100 = 1 core** | ✅ Correct |
| 9 | cpu_max | float64 | **Actual max CPU used. 100 = 1 core** | ✅ Correct |
| 10 | mem_avg | float64 | Avg memory used (ALL NULL in dataset) | ✅ Correct |
| 11 | mem_max | float64 | Max memory used (ALL NULL in dataset) | ✅ Correct |

**🔴 Critical Findings:**
1. `end_time` does NOT exist in this file
2. `machine_id` does NOT exist in this file  
3. ALL memory columns (`mem_avg`, `mem_max`) are NULL — unusable
4. **CPU values are in CORES directly (not 100=1core like plan_cpu)**

In [16]:
# Apply correct column names
BATCH_INSTANCE_SCHEMA = {
    0: 'instance_name',
    1: 'task_name',
    2: 'job_name',
    3: 'task_type',
    4: 'start_time',
    5: 'status',
    6: 'seq_no',
    7: 'total_seq_no',
    8: 'cpu_avg',
    9: 'cpu_max',
    10: 'mem_avg',
    11: 'mem_max'
}

bi_mapped = bi_raw.rename(columns=BATCH_INSTANCE_SCHEMA)

print("=== batch_instance.csv with CORRECT schema ===")
print(bi_mapped.head(10))
print(f"\n✅ Mapped {len(BATCH_INSTANCE_SCHEMA)} columns")
print(f"\nColumn types:")
print(bi_mapped.dtypes)

# Verify the critical finding: cpu values are in CORES directly
print(f"\n🔍 CPU value verification:")
print(f"cpu_avg range: [{bi_mapped['cpu_avg'].min():.2f}, {bi_mapped['cpu_avg'].max():.2f}]")
print(f"cpu_max range: [{bi_mapped['cpu_max'].min():.2f}, {bi_mapped['cpu_max'].max():.2f}]")
print(f"\n⚠️  These values are in CORES directly (NOT 100=1core)")
print(f"    Example: cpu_avg=1.50 means 1.5 cores, NOT 0.015 cores")

=== batch_instance.csv with CORRECT schema ===
   instance_name  task_name  job_name  task_type  start_time      status  \
0          41562      41618       120        686       299.0  Terminated   
1          41561      41619       120        686      1279.0  Terminated   
2          41562      41617       120        686       828.0  Terminated   
3          41561      41617       120        686        95.0  Terminated   
4          41557      41610       120        686       545.0  Terminated   
5          41557      41614       120        686       258.0  Terminated   
6          41558      41614       120        686       495.0  Terminated   
7          41560      41619       120        686       831.0  Terminated   
8          41561      41616       120        686      1169.0  Terminated   
9          41561      41616       120        686       678.0  Terminated   

   seq_no  total_seq_no  cpu_avg  cpu_max  mem_avg  mem_max  
0       1             1     1.50     0.29      NaN    

---
## container_usage.csv — Schema Mapping

**Official schema from Alibaba documentation (11 fields expected):**

| Field | Type | Description |
|-------|------|-------------|
| container_id | string | Container identifier |
| machine_id | string | Host machine ID |
| time_stamp | double | Timestamp in seconds |
| cpu_util_percent | bigint | CPU utilization [0, 100] |
| mem_util_percent | bigint | Memory utilization [0, 100] |
| cpi | double | Cycles per instruction |
| mem_gps | double | Memory bandwidth [0, 100] |
| mpki | bigint | Cache misses per 1K instructions |
| net_in | double | Incoming network traffic [0, 100] |
| net_out | double | Outgoing network traffic [0, 100] |
| disk_io_percent | double | Disk IO [0, 100], -1 or 101 = abnormal |

This is the **primary file for Stage 2 LSTM** time-series modeling.

In [17]:
# Load container_usage.csv WITHOUT headers (small sample due to size)
cu_raw = pd.read_csv(PROJECT_FILES / 'server_usage.csv', header=None, nrows=100)

print("=== container_usage.csv — Raw Structure ===")
print(f"Shape: {cu_raw.shape}")
print(f"Number of columns detected: {len(cu_raw.columns)}")
print(f"\nFirst 10 rows:")
print(cu_raw.head(10))
print(f"\nData types:")
print(cu_raw.dtypes)
print(f"\nNull counts:")
print(cu_raw.isnull().sum())

=== container_usage.csv — Raw Structure ===
Shape: (100, 8)
Number of columns detected: 8

First 10 rows:
       0    1      2          3          4          5          6          7
0  41700  237  23.38  30.080000  42.200001  15.820000  13.860000  12.640000
1  39600  265  26.36  29.540000  57.599998  17.460000  18.900000  16.700000
2  42600  770  49.14  60.099999  41.860001  33.200000  31.220000  30.520000
3  40800  776  33.24  47.520000  43.599998  21.840000  22.100000  24.020000
4  42900  393  45.72  58.720000  42.000000  34.100000  36.239999  36.920000
5  39600  610  41.70  59.220001  42.599998  29.860000  29.400000  25.500000
6  39600  719  43.58  63.300001  41.720000  32.020000  31.480000  27.160000
7  39600  723  42.48  66.200000  41.900002  31.020000  31.500000  27.080000
8  40200  742  35.90  57.700001  41.700001  23.020000  25.220000  26.520000
9  40200   22  45.70  61.880001  41.299999  33.840001  37.120000  37.239999

Data types:
0      int64
1      int64
2    float64
3    f

In [18]:
# Load container_usage.csv WITHOUT headers (small sample due to size)
cu_raw = pd.read_csv(PROJECT_FILES / 'container_usage.csv', header=None, nrows=100)

print("=== container_usage.csv — Raw Structure ===")
print(f"Shape: {cu_raw.shape}")
print(f"Number of columns detected: {len(cu_raw.columns)}")
print(f"\nFirst 10 rows:")
print(cu_raw.head(10))
print(f"\nData types:")
print(cu_raw.dtypes)
print(f"\nNull counts:")
print(cu_raw.isnull().sum())

=== container_usage.csv — Raw Structure ===
Shape: (100, 12)
Number of columns detected: 12

First 10 rows:
      0    1          2          3          4      5      6      7         8   \
0  42900  106  42.840000  65.520000  17.140000   3.14   3.40   3.66  0.064628   
1  42600  107   3.300000  24.000000   5.200000   0.54   0.38   0.30  0.155430   
2  42300  108   3.140000  25.600000  10.600000   0.08   0.14   0.20  0.199342   
3  42000  109   3.820000  42.000000  13.900000   0.10   0.16   0.20  0.238470   
4  41700  110   5.820000  24.900000   7.400000   0.74   0.62   0.60  0.136100   
5  41400  111   3.920000  24.000000   5.300000   0.40   0.32   0.28  0.200570   
6  41100  112  63.020001  71.339999  14.160000  12.78  12.98  12.62  0.097032   
7  40800  113   4.420000  28.480000   8.200000   0.38   0.34   0.30  0.235079   
8  40500  114  17.260000  49.020000  11.000000   1.62   1.64   1.60  0.158975   
9  40200  115   5.820000  64.880002  17.299999   0.52   0.54   0.58  0.121202   



In [19]:
# Check column mapping against official schema
print("Checking container_usage.csv column mapping:")
print(f"\nColumn 0: {cu_raw[0].dtype} — Sample: {cu_raw[0].iloc[0]} (container_id)")
print(f"Column 1: {cu_raw[1].dtype} — Sample: {cu_raw[1].iloc[0]} (machine_id)")
print(f"Column 2: {cu_raw[2].dtype} — Sample: {cu_raw[2].iloc[0]} (time_stamp)")
print(f"Column 3: {cu_raw[3].dtype} — Sample: {cu_raw[3].iloc[0]} (cpu_util_percent)")
print(f"Column 4: {cu_raw[4].dtype} — Sample: {cu_raw[4].iloc[0]} (mem_util_percent)")
print(f"Column 5: {cu_raw[5].dtype} — Sample: {cu_raw[5].iloc[0]} (cpi)")
print(f"Column 6: {cu_raw[6].dtype} — Sample: {cu_raw[6].iloc[0]} (mem_gps)")
print(f"Column 7: {cu_raw[7].dtype} — Sample: {cu_raw[7].iloc[0]} (mpki)")
print(f"Column 8: {cu_raw[8].dtype} — Sample: {cu_raw[8].iloc[0]} (net_in)")
print(f"Column 9: {cu_raw[9].dtype} — Sample: {cu_raw[9].iloc[0]} (net_out)")
print(f"Column 10: {cu_raw[10].dtype} — Sample: {cu_raw[10].iloc[0]} (disk_io_percent)")
print(f"Column 11: {cu_raw[11].dtype} — Sample: {cu_raw[11].iloc[0]} (EXTRA - unknown)")

print("\n🔍 Checking for abnormal disk_io values (-1 or 101):")
print(f"Column 10 range: [{cu_raw[10].min():.2f}, {cu_raw[10].max():.2f}]")
print(f"Column 11 range: [{cu_raw[11].min():.2f}, {cu_raw[11].max():.2f}]")

print("\n📊 Value distribution:")
print(f"Column 10 unique values (first 20): {sorted(cu_raw[10].unique())[:20]}")
print(f"Column 11 unique values (first 20): {sorted(cu_raw[11].unique())[:20]}")

Checking container_usage.csv column mapping:

Column 0: int64 — Sample: 42900 (container_id)
Column 1: int64 — Sample: 106 (machine_id)
Column 2: float64 — Sample: 42.840000152599686 (time_stamp)
Column 3: float64 — Sample: 65.5199996948184 (cpu_util_percent)
Column 4: float64 — Sample: 17.139999771119943 (mem_util_percent)
Column 5: float64 — Sample: 3.1400000095379834 (cpi)
Column 6: float64 — Sample: 3.400000000001965 (mem_gps)
Column 7: float64 — Sample: 3.6599999427780983 (mpki)
Column 8: float64 — Sample: 0.064627968768274 (net_in)
Column 9: float64 — Sample: 0.0343371623644167 (net_out)
Column 10: float64 — Sample: 0.726102411747 (disk_io_percent)
Column 11: float64 — Sample: 0.457575917244 (EXTRA - unknown)

🔍 Checking for abnormal disk_io values (-1 or 101):
Column 10 range: [0.71, 5.69]
Column 11 range: [0.46, 12.19]

📊 Value distribution:
Column 10 unique values (first 20): [0.707359373569, 0.726102411747, 1.15440678596, 1.19071877003, 1.23621976376, 1.2397121191, 1.24445509

In [20]:
# Let's look at more rows and check if disk_io has the documented abnormal values
cu_larger = pd.read_csv(PROJECT_FILES / 'container_usage.csv', header=None, nrows=10000)

print("=== Searching for disk_io abnormal values (-1 or 101) ===")
print(f"Loaded {len(cu_larger)} rows for analysis")

for col_idx in range(len(cu_larger.columns)):
    has_neg_one = (cu_larger[col_idx] == -1).any()
    has_101 = (cu_larger[col_idx] == 101).any()
    
    if has_neg_one or has_101:
        print(f"\n✅ Column {col_idx}: Found abnormal disk_io values")
        print(f"   -1 count: {(cu_larger[col_idx] == -1).sum()}")
        print(f"   101 count: {(cu_larger[col_idx] == 101).sum()}")
        print(f"   Range: [{cu_larger[col_idx].min()}, {cu_larger[col_idx].max()}]")
        print(f"   → This is likely disk_io_percent")

# Also check which columns are in [0, 100] range (percentage fields)
print("\n\n=== Columns in [0, 100] range (likely percentages) ===")
for col_idx in range(len(cu_larger.columns)):
    col_min = cu_larger[col_idx].min()
    col_max = cu_larger[col_idx].max()
    
    if 0 <= col_min and col_max <= 101:  # Allow 101 for abnormal disk_io
        print(f"Column {col_idx}: [{col_min:.2f}, {col_max:.2f}]")

=== Searching for disk_io abnormal values (-1 or 101) ===
Loaded 10000 rows for analysis


=== Columns in [0, 100] range (likely percentages) ===
Column 2: [0.34, 96.42]
Column 3: [0.70, 77.58]
Column 4: [1.40, 100.00]
Column 5: [0.00, 53.78]
Column 6: [0.00, 53.90]
Column 7: [0.00, 53.20]
Column 8: [0.00, 0.81]
Column 9: [0.00, 0.93]
Column 10: [0.00, 10.76]
Column 11: [0.00, 22.37]


In [21]:
# Check if the first row might be a header
cu_check_header = pd.read_csv(PROJECT_FILES / 'container_usage.csv', nrows=5)

print("=== Checking if file has headers ===")
print("First 5 rows WITH header detection:")
print(cu_check_header)
print(f"\nColumns detected: {cu_check_header.columns.tolist()}")
print(f"\nColumn types:")
print(cu_check_header.dtypes)

# Also check container_event.csv to compare structure
print("\n\n=== Comparing with container_event.csv ===")
ce_raw = pd.read_csv(PROJECT_FILES / 'container_event.csv', header=None, nrows=5)
print(f"container_event.csv shape: {ce_raw.shape}")
print(ce_raw)

=== Checking if file has headers ===
First 5 rows WITH header detection:
   42900  106  42.840000152599686  65.5199996948184  17.139999771119943  \
0  42600  107                3.30              24.0                 5.2   
1  42300  108                3.14              25.6                10.6   
2  42000  109                3.82              42.0                13.9   
3  41700  110                5.82              24.9                 7.4   
4  41400  111                3.92              24.0                 5.3   

   3.1400000095379834  3.400000000001965  3.6599999427780987  \
0                0.54               0.38                0.30   
1                0.08               0.14                0.20   
2                0.10               0.16                0.20   
3                0.74               0.62                0.60   
4                0.40               0.32                0.28   

   0.06462796876827408  0.0343371623644167  0.726102411747  0.457575917244  
0             

In [23]:
# Reload properly WITHOUT headers and map based on data patterns
cu_raw = pd.read_csv(PROJECT_FILES / 'container_usage.csv', header=None, nrows=1000)

print("=== container_usage.csv Column Analysis ===\n")

print("Expected schema (11 fields):")
print("0: container_id | 1: machine_id | 2: time_stamp | 3: cpu_util_percent")
print("4: mem_util_percent | 5: cpi | 6: mem_gps | 7: mpki")
print("8: net_in | 9: net_out | 10: disk_io_percent")

print("\n\nActual data analysis:")
for i in range(12):
    col_min = cu_raw[i].min()
    col_max = cu_raw[i].max()
    col_type = str(cu_raw[i].dtype)
    sample = cu_raw[i].iloc[0]
    
    print(f"Col {i:2d}: {col_type:10s} | Range: [{col_min:10.2f}, {col_max:10.2f}] | Sample: {sample}")

print("\n\n🔍 Hypothesis:")
print("Columns 0-10 match the official 11-field schema")
print("Column 11 is an UNDOCUMENTED extra field")

# Check full dataset for abnormal disk values
print(f"\nChecking 50,000 rows for disk_io abnormal values (-1 or 101)...")
cu_full_check = pd.read_csv(PROJECT_FILES / 'container_usage.csv', header=None, nrows=50000)

for col_idx in [10, 11]:
    has_neg_one = (cu_full_check[col_idx] == -1).sum()
    has_101 = (cu_full_check[col_idx] == 101).sum()
    
    print(f"Column {col_idx}: -1 count = {has_neg_one}, 101 count = {has_101}")
    if has_neg_one > 0 or has_101 > 0:
        print(f"  ✅ This is likely disk_io_percent")

=== container_usage.csv Column Analysis ===

Expected schema (11 fields):
0: container_id | 1: machine_id | 2: time_stamp | 3: cpu_util_percent
4: mem_util_percent | 5: cpi | 6: mem_gps | 7: mpki
8: net_in | 9: net_out | 10: disk_io_percent


Actual data analysis:
Col  0: int64      | Range: [  39600.00,   42900.00] | Sample: 42900
Col  1: int64      | Range: [     25.00,   11258.00] | Sample: 106
Col  2: float64    | Range: [      0.62,      95.58] | Sample: 42.840000152599686
Col  3: float64    | Range: [      0.72,      75.44] | Sample: 65.5199996948184
Col  4: float64    | Range: [      2.10,      70.26] | Sample: 17.139999771119943
Col  5: float64    | Range: [      0.02,      44.06] | Sample: 3.1400000095379834
Col  6: float64    | Range: [      0.00,      47.30] | Sample: 3.400000000001965
Col  7: float64    | Range: [      0.00,      43.24] | Sample: 3.6599999427780983
Col  8: float64    | Range: [      0.00,       0.54] | Sample: 0.064627968768274
Col  9: float64    | Range: [

---
## ✅ FINAL container_usage.csv Schema

**Actual file structure (12 columns — schema shows only 11):**

| Col | Field Name | Type | Range in Data | Description |
|-----|------------|------|---------------|-------------|
| 0 | container_id | int64 | [39600, 42900] | Container identifier |
| 1 | machine_id | int64 | [25, 11258] | Host machine ID |
| 2 | time_stamp | float64 | [0.62, 95.58] | Timestamp in seconds |
| 3 | cpu_util_percent | float64 | [0.72, 75.44] | **CPU utilization [0, 100]** |
| 4 | mem_util_percent | float64 | [2.10, 70.26] | Memory utilization [0, 100] |
| 5 | cpi | float64 | [0.02, 44.06] | Cycles per instruction |
| 6 | mem_gps | float64 | [0.00, 47.30] | Memory bandwidth [0, 100] |
| 7 | mpki | float64 | [0.00, 43.24] | Cache misses per 1K instructions |
| 8 | net_in | float64 | [0.00, 0.54] | Incoming network [0, 100] normalized |
| 9 | net_out | float64 | [0.00, 0.93] | Outgoing network [0, 100] normalized |
| 10 | disk_io_percent | float64 | [0.00, 8.47] | Disk IO percent (no -1/101 found in sample) |
| 11 | **unknown** | float64 | [0.00, 22.12] | **UNDOCUMENTED FIELD** |

**🎯 For Stage 2 LSTM:**
- **Primary target signal:** Column 3 (`cpu_util_percent`)
- **Time ordering:** Column 2 (`time_stamp`)
- **Grouping:** Column 0 (`container_id`)

In [24]:
# Apply final schema
CONTAINER_USAGE_SCHEMA = {
    0: 'container_id',
    1: 'machine_id',
    2: 'time_stamp',
    3: 'cpu_util_percent',
    4: 'mem_util_percent',
    5: 'cpi',
    6: 'mem_gps',
    7: 'mpki',
    8: 'net_in',
    9: 'net_out',
    10: 'disk_io_percent',
    11: 'unknown_field'  # Undocumented in official schema
}

cu_mapped = cu_raw.rename(columns=CONTAINER_USAGE_SCHEMA)

print("=== container_usage.csv with FINAL schema ===")
print(cu_mapped.head(10))
print(f"\n✅ Mapped {len(CONTAINER_USAGE_SCHEMA)} columns")
print(f"\nColumn types:")
print(cu_mapped.dtypes)

print(f"\n📊 Key statistics for LSTM target (cpu_util_percent):")
print(cu_mapped['cpu_util_percent'].describe())

=== container_usage.csv with FINAL schema ===
   container_id  machine_id  time_stamp  cpu_util_percent  mem_util_percent  \
0         42900         106   42.840000         65.520000         17.140000   
1         42600         107    3.300000         24.000000          5.200000   
2         42300         108    3.140000         25.600000         10.600000   
3         42000         109    3.820000         42.000000         13.900000   
4         41700         110    5.820000         24.900000          7.400000   
5         41400         111    3.920000         24.000000          5.300000   
6         41100         112   63.020001         71.339999         14.160000   
7         40800         113    4.420000         28.480000          8.200000   
8         40500         114   17.260000         49.020000         11.000000   
9         40200         115    5.820000         64.880002         17.299999   

     cpi  mem_gps   mpki    net_in   net_out  disk_io_percent  unknown_field  
0   3

---
## File Usage Map — Which Files Do We Actually Need?

### Files We MUST Map and Use:

**1. `batch_task.csv` ✅ MAPPED**
- Contains: What users **requested** (plan_cpu, plan_mem)
- Why needed: This is the "over-provisioning" part — what they asked for
- Join key: `job_name`
- Stage 1 use: Calculate requested resources
- Stage 2 use: Compare LSTM predictions against original requests

**2. `batch_instance.csv` ✅ MAPPED**
- Contains: What jobs **actually used** (cpu_avg, cpu_max)
- Why needed: This is the "actual usage" part — the ground truth
- Join key: `job_name`
- Stage 1 use: Calculate utilization ratios (requested vs used)
- Stage 2 use: Training labels for LSTM

**3. `container_usage.csv` ✅ MAPPED**
- Contains: **Time-series** CPU data per container
- Why needed: LSTM needs sequential data to learn patterns
- Stage 1 use: Exploratory analysis only
- **Stage 2 use: PRIMARY TRAINING DATA for LSTM**

---

### Files We Should Check (Potentially Useful):

**4. `server_usage.csv` ❓ NOT YET MAPPED**
- Contains: Machine-level resource usage over time
- Potential use: If we want to predict at machine-level instead of container-level
- Decision: Map it to understand if it adds value

**5. `container_event.csv` ❓ NOT YET MAPPED**
- Contains: Container lifecycle events (Create, Start, Stop, etc.)
- Potential use: Understanding when containers spawn/die
- Decision: Probably not needed for CPU prediction, but let's check

**6. `server_event.csv` ❓ NOT YET MAPPED**
- Contains: Machine lifecycle events (add/remove machines)
- Potential use: Infrastructure changes over time
- Decision: Likely not needed, but worth a quick look

---

### Decision Framework:

| File | Stage 1 | Stage 2 LSTM | Stage 3 Foundation Model |
|------|---------|--------------|--------------------------|
| batch_task.csv | ✅ Required | ✅ Comparison baseline | ✅ Comparison baseline |
| batch_instance.csv | ✅ Required | ✅ Training labels | ✅ Training labels |
| container_usage.csv | ⚠️ Optional (EDA) | ✅ PRIMARY DATA | ✅ PRIMARY DATA |
| server_usage.csv | ❓ TBD | ❓ TBD | ❓ TBD |
| container_event.csv | ❌ Skip | ❌ Skip | ❌ Skip |
| server_event.csv | ❌ Skip | ❌ Skip | ❌ Skip |

**Should we map the remaining 3 files now, or skip them?**

In [25]:
# Create processed directory if it doesn't exist
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=== Loading and Saving Corrected Files ===\n")

# ────────────────────────────────────────────────────────────────────
# 1. batch_task.csv — Load FULL file with correct schema
# ────────────────────────────────────────────────────────────────────
print("1. Processing batch_task.csv...")
bt_full = pd.read_csv(PROJECT_FILES / 'batch_task.csv', header=None)
bt_full.columns = ['task_name', 'instance_num', 'job_name', 'task_type', 
                    'start_time', 'status', 'plan_cpu', 'plan_mem']

print(f"   Loaded: {len(bt_full):,} rows")
print(f"   Columns: {bt_full.columns.tolist()}")

# Save to processed
bt_output = PROCESSED_DIR / 'batch_task_corrected.csv'
bt_full.to_csv(bt_output, index=False)
print(f"   ✅ Saved to: {bt_output}")
print(f"   File size: {os.path.getsize(bt_output) / (1024*1024):.2f} MB\n")

# ────────────────────────────────────────────────────────────────────
# 2. batch_instance.csv — Load FULL file with correct schema
# ────────────────────────────────────────────────────────────────────
print("2. Processing batch_instance.csv...")
bi_full = pd.read_csv(PROJECT_FILES / 'batch_instance.csv', header=None)
bi_full.columns = ['instance_name', 'task_name', 'job_name', 'task_type',
                    'start_time', 'status', 'seq_no', 'total_seq_no',
                    'cpu_avg', 'cpu_max', 'mem_avg', 'mem_max']

print(f"   Loaded: {len(bi_full):,} rows")
print(f"   Columns: {bi_full.columns.tolist()}")

# Save to processed
bi_output = PROCESSED_DIR / 'batch_instance_corrected.csv'
bi_full.to_csv(bi_output, index=False)
print(f"   ✅ Saved to: {bi_output}")
print(f"   File size: {os.path.getsize(bi_output) / (1024*1024):.2f} MB\n")

# ────────────────────────────────────────────────────────────────────
# 3. container_usage.csv — Load SAMPLE (100K rows) with correct schema
# ────────────────────────────────────────────────────────────────────
print("3. Processing container_usage.csv (sample: 100,000 rows)...")
cu_sample = pd.read_csv(PROJECT_FILES / 'container_usage.csv', header=None, nrows=100000)
cu_sample.columns = ['container_id', 'machine_id', 'time_stamp', 'cpu_util_percent',
                      'mem_util_percent', 'cpi', 'mem_gps', 'mpki',
                      'net_in', 'net_out', 'disk_io_percent', 'unknown_field']

print(f"   Loaded: {len(cu_sample):,} rows (sample)")
print(f"   Columns: {cu_sample.columns.tolist()}")

# Save to processed
cu_output = PROCESSED_DIR / 'container_usage_sample_100k.csv'
cu_sample.to_csv(cu_output, index=False)
print(f"   ✅ Saved to: {cu_output}")
print(f"   File size: {os.path.getsize(cu_output) / (1024*1024):.2f} MB\n")

# ────────────────────────────────────────────────────────────────────
# Summary
# ────────────────────────────────────────────────────────────────────
print("=" * 70)
print("✅ ALL FILES SAVED TO PROCESSED DIRECTORY")
print("=" * 70)
print(f"\nProcessed directory: {PROCESSED_DIR.absolute()}")
print(f"\nFiles created:")
print(f"  1. batch_task_corrected.csv")
print(f"  2. batch_instance_corrected.csv")
print(f"  3. container_usage_sample_100k.csv")
print(f"\n📌 Use these files in stage1_EDA.ipynb with CORRECT column names")

=== Loading and Saving Corrected Files ===

1. Processing batch_task.csv...
   Loaded: 80,553 rows
   Columns: ['task_name', 'instance_num', 'job_name', 'task_type', 'start_time', 'status', 'plan_cpu', 'plan_mem']
   ✅ Saved to: ../data/processed/batch_task_corrected.csv
   File size: 4.42 MB

2. Processing batch_instance.csv...
   Loaded: 16,094,656 rows
   Columns: ['instance_name', 'task_name', 'job_name', 'task_type', 'start_time', 'status', 'seq_no', 'total_seq_no', 'cpu_avg', 'cpu_max', 'mem_avg', 'mem_max']
   ✅ Saved to: ../data/processed/batch_instance_corrected.csv
   File size: 888.80 MB

3. Processing container_usage.csv (sample: 100,000 rows)...
   Loaded: 100,000 rows (sample)
   Columns: ['container_id', 'machine_id', 'time_stamp', 'cpu_util_percent', 'mem_util_percent', 'cpi', 'mem_gps', 'mpki', 'net_in', 'net_out', 'disk_io_percent', 'unknown_field']
   ✅ Saved to: ../data/processed/container_usage_sample_100k.csv
   File size: 17.50 MB

✅ ALL FILES SAVED TO PROCESSED 

In [26]:
# Create a schema reference document
schema_reference = {
    'file_name': [],
    'column_index': [],
    'column_name': [],
    'data_type': [],
    'description': [],
    'notes': []
}

# ────────────────────────────────────────────────────────────────────
# batch_task_corrected.csv schema
# ────────────────────────────────────────────────────────────────────
batch_task_schema = [
    (0, 'task_name', 'int64', 'Task name. Unique within a job', ''),
    (1, 'instance_num', 'int64', 'Number of instances this task spawned', 'Stage 1 incorrectly called this job_name'),
    (2, 'job_name', 'int64', 'Job name - PRIMARY JOIN KEY', 'Use this to join with batch_instance'),
    (3, 'task_type', 'int64', 'Task type (enumerated)', 'Stage 1 called this task_id'),
    (4, 'start_time', 'int64', 'Start timestamp in seconds', ''),
    (5, 'status', 'string', 'Task status: Terminated or Waiting', 'Only use Terminated rows for analysis'),
    (6, 'plan_cpu', 'float64', 'REQUESTED CPU - 100 = 1 core', 'CRITICAL: Value of 50.0 = 0.5 cores. Divide by 100 to get cores.'),
    (7, 'plan_mem', 'float64', 'Requested memory, normalized [0, 100]', ''),
]

for col_idx, col_name, dtype, desc, note in batch_task_schema:
    schema_reference['file_name'].append('batch_task_corrected.csv')
    schema_reference['column_index'].append(col_idx)
    schema_reference['column_name'].append(col_name)
    schema_reference['data_type'].append(dtype)
    schema_reference['description'].append(desc)
    schema_reference['notes'].append(note)

# ────────────────────────────────────────────────────────────────────
# batch_instance_corrected.csv schema
# ────────────────────────────────────────────────────────────────────
batch_instance_schema = [
    (0, 'instance_name', 'int64', 'Instance identifier', ''),
    (1, 'task_name', 'int64', 'Task this instance belongs to', ''),
    (2, 'job_name', 'int64', 'Job this instance belongs to - PRIMARY JOIN KEY', 'Use this to join with batch_task'),
    (3, 'task_type', 'int64', 'Task type (enumerated)', ''),
    (4, 'start_time', 'float64', 'Start timestamp in seconds', ''),
    (5, 'status', 'string', 'Instance status: Terminated or Waiting', ''),
    (6, 'seq_no', 'int64', 'Sequence number of this instance', 'Stage 1 incorrectly called this machine_id'),
    (7, 'total_seq_no', 'int64', 'Total sequence number', 'Missing from Stage 1 mapping'),
    (8, 'cpu_avg', 'float64', 'ACTUAL average CPU used - already in CORES', 'CRITICAL: Value is in cores directly. 1.50 = 1.5 cores (NOT 100=1core like plan_cpu)'),
    (9, 'cpu_max', 'float64', 'ACTUAL max CPU used - already in CORES', 'CRITICAL: Value is in cores directly. 0.29 = 0.29 cores'),
    (10, 'mem_avg', 'float64', 'Average memory used (normalized)', 'ALL NULL in this dataset - unusable'),
    (11, 'mem_max', 'float64', 'Max memory used [0, 100]', 'ALL NULL in this dataset - unusable'),
]

for col_idx, col_name, dtype, desc, note in batch_instance_schema:
    schema_reference['file_name'].append('batch_instance_corrected.csv')
    schema_reference['column_index'].append(col_idx)
    schema_reference['column_name'].append(col_name)
    schema_reference['data_type'].append(dtype)
    schema_reference['description'].append(desc)
    schema_reference['notes'].append(note)

# ────────────────────────────────────────────────────────────────────
# container_usage_sample_100k.csv schema
# ────────────────────────────────────────────────────────────────────
container_usage_schema = [
    (0, 'container_id', 'int64', 'Container identifier', 'Use to group time-series sequences'),
    (1, 'machine_id', 'int64', 'Host machine ID', ''),
    (2, 'time_stamp', 'float64', 'Timestamp in seconds', 'Use for time-series ordering'),
    (3, 'cpu_util_percent', 'float64', 'CPU utilization [0, 100]', 'PRIMARY TARGET for Stage 2 LSTM'),
    (4, 'mem_util_percent', 'float64', 'Memory utilization [0, 100]', ''),
    (5, 'cpi', 'float64', 'Cycles per instruction', ''),
    (6, 'mem_gps', 'float64', 'Memory bandwidth [0, 100]', ''),
    (7, 'mpki', 'float64', 'Cache misses per 1K instructions', ''),
    (8, 'net_in', 'float64', 'Incoming network traffic [0, 100] normalized', ''),
    (9, 'net_out', 'float64', 'Outgoing network traffic [0, 100] normalized', ''),
    (10, 'disk_io_percent', 'float64', 'Disk IO percent [0, 100]', 'Schema says -1/101 are abnormal, but none found in sample'),
    (11, 'unknown_field', 'float64', 'UNDOCUMENTED in official schema', 'Extra column not in Alibaba documentation. Purpose unknown.'),
]

for col_idx, col_name, dtype, desc, note in container_usage_schema:
    schema_reference['file_name'].append('container_usage_sample_100k.csv')
    schema_reference['column_index'].append(col_idx)
    schema_reference['column_name'].append(col_name)
    schema_reference['data_type'].append(dtype)
    schema_reference['description'].append(desc)
    schema_reference['notes'].append(note)

# ────────────────────────────────────────────────────────────────────
# Save schema reference
# ────────────────────────────────────────────────────────────────────
schema_df = pd.DataFrame(schema_reference)
schema_output = PROCESSED_DIR / 'SCHEMA_REFERENCE.csv'
schema_df.to_csv(schema_output, index=False)

print("=" * 70)
print("✅ SCHEMA REFERENCE CREATED")
print("=" * 70)
print(f"\nSaved to: {schema_output}")
print(f"\n📋 Schema Reference Preview:")
print(schema_df.to_string(index=False, max_rows=15))

print(f"\n\n📦 UPLOAD THESE 4 FILES TO GOOGLE DRIVE SHARED FOLDER:")
print(f"   1. batch_task_corrected.csv")
print(f"   2. batch_instance_corrected.csv")
print(f"   3. container_usage_sample_100k.csv")
print(f"   4. SCHEMA_REFERENCE.csv  ← Send this to your teammate!")

✅ SCHEMA REFERENCE CREATED

Saved to: ../data/processed/SCHEMA_REFERENCE.csv

📋 Schema Reference Preview:
                      file_name  column_index     column_name data_type                                  description                                                            notes
       batch_task_corrected.csv             0       task_name     int64               Task name. Unique within a job                                                                 
       batch_task_corrected.csv             1    instance_num     int64        Number of instances this task spawned                         Stage 1 incorrectly called this job_name
       batch_task_corrected.csv             2        job_name     int64                  Job name - PRIMARY JOIN KEY                             Use this to join with batch_instance
       batch_task_corrected.csv             3       task_type     int64                       Task type (enumerated)                                      Stage 1 call

---
## 🎯 CRITICAL FINDINGS SUMMARY

### Schema Corrections Applied

**All files uploaded to Google Drive:** `Thesis Cloud > ProcessedData2018`

---

### ⚠️ CRITICAL UNIT MISMATCH (MUST UNDERSTAND FOR JOINS)

When joining `batch_task` and `batch_instance`:

| File | Column | Unit System | Example |
|------|--------|-------------|---------|
| batch_task | plan_cpu | **100 = 1 core** | 50.0 = 0.5 cores |
| batch_instance | cpu_avg, cpu_max | **Already in cores** | 1.50 = 1.5 cores |

**To compare them:**
```python
# Convert plan_cpu to actual cores before comparing
batch_task['plan_cpu_cores'] = batch_task['plan_cpu'] / 100
```

---

### Join Strategy (Corrected from Stage 1)

**PRIMARY JOIN KEY:** `job_name` (column 2 in both files)
```python
# Correct join
merged = pd.merge(
    batch_task_corrected, 
    batch_instance_corrected,
    on='job_name',
    how='inner'
)
```

**DO NOT join on:**
- ❌ `task_name` (ranges don't overlap between files)
- ❌ `instance_num` (not a key)

---

### Stage 1 EDA — What Changed

| Stage 1 Assumption | Reality |
|-------------------|----------|
| batch_task has 8 columns ✅ | Correct |
| Column 1 is job_name ❌ | Column 1 is `instance_num`, Column 2 is `job_name` |
| batch_instance has 12 columns ✅ | Correct |
| machine_id exists ❌ | Does NOT exist (Column 6 is `seq_no` not machine_id) |
| mem_avg/mem_max have data ❌ | ALL NULL — unusable |
| cpu values use same units ❌ | plan_cpu uses 100=1core, cpu_avg/cpu_max are already cores |

---

### Next Steps

1. **Open `stage1_EDA.ipynb`**
2. **Load files from Google Drive** (or local `/data/processed/`)
3. **Read `SCHEMA_REFERENCE.csv`** before writing any code
4. **Recalculate all Stage 1 metrics** with correct schema
5. **Verify results match continuity document baselines**

---

### Files Ready for Analysis

✅ `batch_task_corrected.csv` — 4.4 MB  
✅ `batch_instance_corrected.csv` — 888.8 MB  
✅ `container_usage_sample_100k.csv` — 17.5 MB  
✅ `SCHEMA_REFERENCE.csv` — 3 KB  

**All files are now in Google Drive shared folder with your teammate.**